# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.patches import Patch, Rectangle
from matplotlib.transforms import blended_transform_factory

semantic_thresholds = {
    "Near identical": 0.9,
    "Strong conceptual similarity": 0.7,
    "Partial overlap": 0.5,
    "Low / no semantic relevance": 0.0
}

# Distribution of similarity scores within a single LLM

In [ ]:
PROCESS_COLOR   = "#000000"
NARRATIVE_COLOR = "#B8C4CE"
BOX_COLOR_PROC  = "#4487CF"
BOX_COLOR_NARR  = "#4487CF"
VIOLIN_WIDTH    = 0.38
BOX_HW          = 0.03
GAP             = 0.08

BENCHMARK_SETS_WITHIN = ["full_set", "reduced_set", "noise_20", "noise_40", "noise_60", "noise_80"]
BENCHMARK_LABELS_WITHIN = {
    "full_set":    "Full Set",
    "reduced_set": "Pharmacologically\nActionable Set",
    "noise_20":    "Noise 20%",
    "noise_40":    "Noise 40%",
    "noise_60":    "Noise 60%",
    "noise_80":    "Noise 80%",
}

def _plot_within_llm(ax, df_proc, df_narr, panel_label="", title="", show_threshold_labels=False):
    """
    Draws split half-violins per benchmark set onto ax, with a gap at center:
      left  — process name (dark blue) with its own gold box plot
      right — analytic narrative (light gray) with its own gold box plot
    """
    half_gap = GAP / 2

    for i, bset in enumerate(BENCHMARK_SETS_WITHIN):
        proc_vals = df_proc[df_proc["prediction_type"] == bset]["semantic_similarity"].dropna().values
        narr_vals = df_narr[df_narr["prediction_type"] == bset]["semantic_similarity"].dropna().values

        # Left half-violin + box: process names
        if len(proc_vals) > 1:
            kde = stats.gaussian_kde(proc_vals, bw_method=0.25)
            y   = np.linspace(proc_vals.min(), proc_vals.max(), 300)
            d   = kde(y)
            d   = d / d.max() * VIOLIN_WIDTH
            ax.fill_betweenx(y, i - half_gap - d, i - half_gap, color=PROCESS_COLOR, alpha=0.80)

            q1, med, q3 = np.percentile(proc_vals, [25, 50, 75])
            iqr = q3 - q1
            lo  = max(proc_vals.min(), q1 - 1.5 * iqr)
            hi  = min(proc_vals.max(), q3 + 1.5 * iqr)
            cx  = i - half_gap
            ax.plot([cx, cx], [lo, hi], color=BOX_COLOR_PROC, lw=2.0, zorder=3)
            ax.add_patch(Rectangle((cx - BOX_HW, q1), 2 * BOX_HW, iqr, color=BOX_COLOR_PROC, zorder=4))
            ax.plot(cx, med, "o", color="white", ms=6, zorder=5)

        # Right half-violin + box: analytic narratives
        if len(narr_vals) > 1:
            kde = stats.gaussian_kde(narr_vals, bw_method=0.25)
            y   = np.linspace(narr_vals.min(), narr_vals.max(), 300)
            d   = kde(y)
            d   = d / d.max() * VIOLIN_WIDTH
            ax.fill_betweenx(y, i + half_gap, i + half_gap + d, color=NARRATIVE_COLOR, alpha=0.80)

            q1, med, q3 = np.percentile(narr_vals, [25, 50, 75])
            iqr = q3 - q1
            lo  = max(narr_vals.min(), q1 - 1.5 * iqr)
            hi  = min(narr_vals.max(), q3 + 1.5 * iqr)
            cx  = i + half_gap
            ax.plot([cx, cx], [lo, hi], color=BOX_COLOR_NARR, lw=2.0, zorder=3)
            ax.add_patch(Rectangle((cx - BOX_HW, q1), 2 * BOX_HW, iqr, color=BOX_COLOR_NARR, zorder=4))
            ax.plot(cx, med, "o", color="white", ms=6, zorder=5)

    ax.set_xticks(range(len(BENCHMARK_SETS_WITHIN)))
    ax.set_xticklabels([BENCHMARK_LABELS_WITHIN[b] for b in BENCHMARK_SETS_WITHIN], fontsize=10)
    ax.set_ylim(0.2, 1.05)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if title:
        ax.set_title(title, fontsize=13)
    if panel_label:
        ax.text(-0.08, 1.0, panel_label, transform=ax.transAxes,
                fontsize=16, fontweight="bold", va="bottom", ha="left")

    # Semantic threshold lines and band labels (just right of the axes)
    threshold_items = list(semantic_thresholds.items())
    y_min_vis, y_max_vis = ax.get_ylim()

    bands = []
    for k, (label, lower) in enumerate(threshold_items):
        upper = threshold_items[k - 1][1] if k > 0 else y_max_vis
        bands.append((label, lower, upper))

    for label, lower, upper in bands:
        if lower >= y_min_vis:
            ax.axhline(lower, color="gray", lw=0.9, ls="--", zorder=1, alpha=0.7)

    if show_threshold_labels:
        trans = blended_transform_factory(ax.transAxes, ax.transData)
        for label, lower, upper in bands:
            mid_y = (max(lower, y_min_vis) + min(upper, y_max_vis)) / 2
            ax.text(
                1.02, mid_y, label,
                transform=trans,
                ha="left", va="center", fontsize=8.5,
                color="dimgray", style="italic",
                bbox=dict(boxstyle="round,pad=0.25", fc="#e8e8e8", ec="none", alpha=0.85),
            )


def draw_within_llm_distribution(
    llm_csv_paths: dict,
    output_path: str = "figures/within_llm_distribution.png",
):
    """
    llm_csv_paths: {llm_label: (processname_csv, narrative_csv)}
    Draws one panel per LLM in a 2x2 grid (labeled A-D), each showing split
    half-violins (process name vs analytic narrative) per benchmark set, with
    a single legend shared across all panels. Semantic-similarity threshold
    bands are labeled only on the rightmost column to avoid clutter.
    """
    llm_labels = list(llm_csv_paths.keys())
    panel_labels = ["A", "B", "C", "D"]
    n_cols = 2
    fig, axes = plt.subplots(2, n_cols, figsize=(15, 9.5), sharey=True)

    for idx, (panel_label, ax, llm_label) in enumerate(zip(panel_labels, axes.flat, llm_labels)):
        processname_csv, narrative_csv = llm_csv_paths[llm_label]
        df_proc = pd.read_csv(processname_csv)
        df_narr = pd.read_csv(narrative_csv)
        is_rightmost = (idx % n_cols) == (n_cols - 1)
        _plot_within_llm(ax, df_proc, df_narr, panel_label=panel_label, title=llm_label,
                          show_threshold_labels=is_rightmost)

    for ax in axes[:, 0]:
        ax.set_ylabel("Semantic Similarity", fontsize=12)
    for ax in axes[-1, :]:
        ax.set_xlabel("Benchmark Set", fontsize=12)

    legend_handles = [
        Patch(facecolor=PROCESS_COLOR,   alpha=0.8, label="Process Name"),
        Patch(facecolor=NARRATIVE_COLOR, alpha=0.8, label="Analytic Narrative"),
    ]
    fig.legend(handles=legend_handles, frameon=False, fontsize=11,
               loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
LLM_BASE_PATHS = {
    "GPT-5.4 mini": "Outputs/azure-gpt-5.4-mini",
    "GPT-OSS":       "Outputs/gpt-oss:20b",
    "Gemma4":        "Outputs/gemma4:26b",
    "Mixtral":       "Outputs/mixtral-8x22b",
}

llm_csv_paths = {
    llm_label: (
        f"{base_path}/evaluation_results_processNames.csv",
        f"{base_path}/evaluation_results_analyticNarratives.csv",
    )
    for llm_label, base_path in LLM_BASE_PATHS.items()
}
draw_within_llm_distribution(llm_csv_paths, output_path="figures/within_llm_distribution.png")

# Distribution of LLM judge scores within a single LLM

In [ ]:
LLM_JUDGE_SCORE_ORDER = [1, 2, 3, 4]
LLM_JUDGE_COLORS_HIST = ["#ffffcc", "#d9f0a3", "#78c679", "#1a9641"]

def _plot_within_llm_judge_scores(ax, df_proc, df_narr, panel_label="", title=""):
    """
    Draws two adjacent stacked bars per benchmark set onto ax:
      left  bar (outlined in process color)   — process name judge score proportions
      right bar (outlined in narrative color) — analytic narrative judge score proportions
    """
    bar_w = 0.32
    gap   = 0.06
    x     = np.arange(len(BENCHMARK_SETS_WITHIN))

    for df, sign, edge_color in [(df_proc, -1, PROCESS_COLOR), (df_narr, 1, NARRATIVE_COLOR)]:
        df = df.dropna(subset=["llm_judge_score"]).copy()
        df["llm_score"] = df["llm_judge_score"].astype(int)
        bottoms = np.zeros(len(BENCHMARK_SETS_WITHIN))
        for score, color in zip(LLM_JUDGE_SCORE_ORDER, LLM_JUDGE_COLORS_HIST):
            heights = [
                df[df["prediction_type"] == bset]["llm_score"].eq(score).sum()
                / max(df[df["prediction_type"] == bset]["llm_score"].count(), 1)
                for bset in BENCHMARK_SETS_WITHIN
            ]
            ax.bar(x + sign * (bar_w / 2 + gap / 2), heights, bar_w, bottom=bottoms,
                   color=color, edgecolor=edge_color, linewidth=1.2)
            bottoms += np.array(heights)

    ax.set_xticks(x)
    ax.set_xticklabels([BENCHMARK_LABELS_WITHIN[b] for b in BENCHMARK_SETS_WITHIN], fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if title:
        ax.set_title(title, fontsize=13)
    if panel_label:
        ax.text(-0.08, 1.0, panel_label, transform=ax.transAxes,
                fontsize=16, fontweight="bold", va="bottom", ha="left")


def draw_within_llm_judge_score_distribution(
    llm_csv_paths: dict,
    output_path: str = "figures/within_llm_judge_score_distribution.png",
):
    """
    llm_csv_paths: {llm_label: (processname_csv, narrative_csv)}
    Draws one panel per LLM in a 2x2 grid (labeled A-D), each showing stacked
    bars of LLM judge score proportions per benchmark set:
      left  bar — process name
      right bar — analytic narrative
    A single legend below the figure applies to all panels.
    """
    llm_labels = list(llm_csv_paths.keys())
    panel_labels = ["A", "B", "C", "D"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 9.5), sharey=True)

    for panel_label, ax, llm_label in zip(panel_labels, axes.flat, llm_labels):
        processname_csv, narrative_csv = llm_csv_paths[llm_label]
        df_proc = pd.read_csv(processname_csv)
        df_narr = pd.read_csv(narrative_csv)
        _plot_within_llm_judge_scores(ax, df_proc, df_narr, panel_label=panel_label, title=llm_label)

    for ax in axes[:, 0]:
        ax.set_ylabel("Proportion", fontsize=12)
    for ax in axes[-1, :]:
        ax.set_xlabel("Benchmark Set", fontsize=12)

    score_handles = [Patch(facecolor=c, label=f"Score {s}") for s, c in zip(LLM_JUDGE_SCORE_ORDER, LLM_JUDGE_COLORS_HIST)]
    score_handles.reverse()
    task_handles = [
        Patch(facecolor="white", edgecolor=PROCESS_COLOR,   lw=1.5, label="Process Name"),
        Patch(facecolor="white", edgecolor=NARRATIVE_COLOR, lw=1.5, label="Analytic Narrative"),
    ]
    fig.legend(handles=score_handles + task_handles, frameon=False, fontsize=10,
               loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=len(score_handles) + len(task_handles))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
draw_within_llm_judge_score_distribution(llm_csv_paths, output_path="figures/within_llm_judge_score_distribution.png")

# Distribution of similarity scores across LLMs

In [ ]:
BENCHMARK_LABELS = {
    "full_set":    "Full Set",
    "reduced_set": "Pharmacologically\nActionable Set",
    "noise_20":    "Noise 20%",
    "noise_40":    "Noise 40%",
    "noise_60":    "Noise 60%",
    "noise_80":    "Noise 80%",
}

def draw_similarity_distribution(llm_data: dict, output_path: str = "similarity_distribution.png", title: str = ""):
    """
    llm_data: {llm_label: {benchmark_set: np.ndarray of semantic_similarity values}}
    Draws one full violin + box per LLM per benchmark set, grouped by benchmark set on the x-axis.
    """
    LLM_COLORS = {
        "GPT-5.4 mini": "#648FFF",
        "GPT-OSS":       "#785EF0",
        "Gemma4":        "#DC267F",
        "Mixtral":       "#FE6100",
    }
    VIOLIN_WIDTH = 0.16
    SUB_SPACING  = 0.22
    BOX_HW       = 0.02

    benchmark_sets = list(next(iter(llm_data.values())).keys())
    llm_labels     = list(llm_data.keys())
    n_llms         = len(llm_labels)
    offsets        = np.linspace(-(n_llms - 1) / 2, (n_llms - 1) / 2, n_llms) * SUB_SPACING

    def full_violin(ax, values, x_center, color, alpha=0.4, bw=0.25):
        kde = stats.gaussian_kde(values, bw_method=bw)
        y   = np.linspace(values.min(), values.max(), 300)
        d   = kde(y)
        d   = d / d.max() * VIOLIN_WIDTH
        ax.fill_betweenx(y, x_center - d / 2, x_center + d / 2, color=color, alpha=alpha)

    def box_plot(ax, values, x_center, color):
        q1, med, q3 = np.percentile(values, [25, 50, 75])
        iqr = q3 - q1
        lo  = max(values.min(), q1 - 1.5 * iqr)
        hi  = min(values.max(), q3 + 1.5 * iqr)
        ax.plot([x_center, x_center], [lo, hi], color=color, lw=1.4, zorder=3)
        ax.add_patch(Rectangle((x_center - BOX_HW, q1), 2 * BOX_HW, iqr, color=color, zorder=4))
        ax.plot(x_center, med, "o", color="white", ms=4, zorder=5)

    fig, ax = plt.subplots(figsize=(12, 5))

    for i, bset in enumerate(benchmark_sets):
        for j, llm in enumerate(llm_labels):
            x      = i + offsets[j]
            values = llm_data[llm][bset]
            color  = LLM_COLORS[llm]
            full_violin(ax, values, x, color)
            box_plot(ax, values, x, color)

    ax.set_xticks(range(len(benchmark_sets)))
    ax.set_xticklabels([BENCHMARK_LABELS.get(b, b) for b in benchmark_sets], fontsize=11)
    ax.set_ylabel("Semantic Similarity", fontsize=12)
    ax.set_xlabel("Benchmark Set", fontsize=12)
    if title:
        ax.set_title(title, fontsize=13)
    ax.set_ylim(0.2, 1.05)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Semantic threshold lines and band labels (just right of the axes)
    threshold_items = list(semantic_thresholds.items())
    y_min_vis, y_max_vis = ax.get_ylim()

    bands = []
    for k, (label, lower) in enumerate(threshold_items):
        upper = threshold_items[k - 1][1] if k > 0 else y_max_vis
        bands.append((label, lower, upper))

    trans = blended_transform_factory(ax.transAxes, ax.transData)

    for label, lower, upper in bands:
        if lower >= y_min_vis:
            ax.axhline(lower, color="gray", lw=0.9, ls="--", zorder=1, alpha=0.7)
        mid_y = (max(lower, y_min_vis) + min(upper, y_max_vis)) / 2
        ax.text(
            1.02, mid_y, label,
            transform=trans,
            ha="left", va="center", fontsize=8.5,
            color="dimgray", style="italic",
            bbox=dict(boxstyle="round,pad=0.25", fc="#e8e8e8", ec="none", alpha=0.85),
        )

    # Color legend centered below the x-axis label
    legend_handles = [Patch(facecolor=LLM_COLORS[l], alpha=0.8, label=l) for l in llm_labels]
    ax.legend(handles=legend_handles, frameon=False, fontsize=10,
              loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=len(llm_labels))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")

## Process Names

In [ ]:
def load_process_name_data(llm_csv_paths: dict, benchmark_sets: list) -> dict:
    """
    llm_csv_paths: {display_label: path_to_evaluation_results_processNames.csv}
    Returns: {display_label: {benchmark_set: np.ndarray of semantic_similarity values}}
    """
    llm_data = {}
    for label, csv_path in llm_csv_paths.items():
        df = pd.read_csv(csv_path)
        llm_data[label] = {
            bset: df[df["prediction_type"] == bset]["semantic_similarity"].dropna().values
            for bset in benchmark_sets
        }
    return llm_data

BENCHMARK_SETS = ["full_set", "reduced_set", "noise_20", "noise_40", "noise_60", "noise_80"]

LLM_CSV_PATHS = {
    "GPT-5.4 mini": "Outputs/azure-gpt-5.4-mini/evaluation_results_processNames.csv",
    "GPT-OSS":       "Outputs/gpt-oss:20b/evaluation_results_processNames.csv",
    "Gemma4":        "Outputs/gemma4:26b/evaluation_results_processNames.csv",
    "Mixtral":       "Outputs/mixtral-8x22b/evaluation_results_processNames.csv",
}

llm_data = load_process_name_data(LLM_CSV_PATHS, BENCHMARK_SETS)
draw_similarity_distribution(llm_data, title="Process Names", output_path="figures/similarity_distribution_processNames.png")

## Analytic Narratives

In [ ]:
def load_process_name_data(llm_csv_paths: dict, benchmark_sets: list) -> dict:
    """
    llm_csv_paths: {display_label: path_to_evaluation_results_analyticNarratives.csv}
    Returns: {display_label: {benchmark_set: np.ndarray of semantic_similarity values}}
    """
    llm_data = {}
    for label, csv_path in llm_csv_paths.items():
        df = pd.read_csv(csv_path)
        llm_data[label] = {
            bset: df[df["prediction_type"] == bset]["semantic_similarity"].dropna().values
            for bset in benchmark_sets
        }
    return llm_data

BENCHMARK_SETS = ["full_set", "reduced_set", "noise_20", "noise_40", "noise_60", "noise_80"]

LLM_CSV_PATHS = {
    "GPT-5.4 mini": "Outputs/azure-gpt-5.4-mini/evaluation_results_analyticNarratives.csv",
    "Gemma4":        "Outputs/gemma4:26b/evaluation_results_analyticNarratives.csv",
    "GPT-OSS":       "Outputs/gpt-oss:20b/evaluation_results_analyticNarratives.csv",
    "Mixtral":       "Outputs/mixtral-8x22b/evaluation_results_analyticNarratives.csv",
}

llm_data = load_process_name_data(LLM_CSV_PATHS, BENCHMARK_SETS)
draw_similarity_distribution(llm_data, title="Analytic Narratives", output_path="figures/similarity_distribution_analyticNarratives.png")

# Sankey: Semantic Similarity → LLM Judge Score

In [ ]:
import plotly.graph_objects as go
from IPython.display import display, HTML

SEM_BINS = [
    "Near identical",
    "Strong conceptual similarity",
    "Partial overlap",
    "Low / no semantic relevance",
]

def bin_semantic(score: float) -> str:
    if score >= 0.9:
        return "Near identical"
    elif score >= 0.7:
        return "Strong conceptual similarity"
    elif score >= 0.5:
        return "Partial overlap"
    else:
        return "Low / no semantic relevance"

def draw_sankey(csv_path: str, noise_level: str, title: str = "", output_filename: str = "sankey"):
    df = pd.read_csv(csv_path).dropna(subset=["semantic_similarity", "llm_judge_score"])
    df = df[df["prediction_type"] == noise_level]
    if df.empty:
        print(f"No data for {noise_level} in {csv_path}")
        return

    df["sem_bin"] = df["semantic_similarity"].apply(bin_semantic)
    df["llm_label"] = df["llm_judge_score"].astype(int).apply(lambda s: f"LLM Score {s}")

    llm_labels = [f"LLM Score {s}" for s in sorted(df["llm_judge_score"].astype(int).unique(), reverse=True)]
    all_labels = SEM_BINS + llm_labels
    idx = {label: i for i, label in enumerate(all_labels)}

    flows = df.groupby(["sem_bin", "llm_label"]).size().reset_index(name="count")
    sources = [idx[r["sem_bin"]]   for _, r in flows.iterrows()]
    targets = [idx[r["llm_label"]] for _, r in flows.iterrows()]
    values  = flows["count"].tolist()

    sem_colors  = ["#2c7bb6", "#74add1", "#fdae61", "#d7191c"]
    llm_colors  = ["#1a9641", "#78c679", "#d9f0a3", "#ffffcc"]
    node_colors = sem_colors + llm_colors[: len(llm_labels)]

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=20, thickness=22,
            line=dict(color="white", width=0.5),
            label=all_labels,
            color=node_colors,
        ),
        link=dict(source=sources, target=targets, value=values),
    ))
    fig.update_layout(
        title_text=title or f"Semantic Similarity → LLM Judge Score ({noise_level})",
        font=dict(size=13),
        margin=dict(l=120, r=120, t=50, b=20),
    )

    fig.write_html(f"{output_filename}.html")
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

In [ ]:
SANKEY_LLM_PATHS = {
    "GPT-5.4 mini": "Outputs/azure-gpt-5.4-mini/evaluation_results_processNames.csv",
    "GPT-OSS":       "Outputs/gpt-oss:20b/evaluation_results_processNames.csv",
    "Gemma4":        "Outputs/gemma4:26b/evaluation_results_processNames.csv",
    "Mixtral":       "Outputs/mixtral-8x22b/evaluation_results_processNames.csv",
}

NOISE_LEVELS = ["reduced_set", "noise_20", "noise_40", "noise_60", "noise_80"]

for llm_label, csv_path in SANKEY_LLM_PATHS.items():
    for noise_level in NOISE_LEVELS:
        llm_slug = llm_label.lower().replace(" ", "-").replace(":", "")
        output_filename = f"figures/sankey_{llm_slug}_{noise_level}"
        draw_sankey(
            csv_path=csv_path,
            noise_level=noise_level,
            title=f"{llm_label} · {noise_level}",
            output_filename=output_filename,
        )

# Stacked histogram: Semantic Similarity vs LLM Judge Score

In [ ]:
# Stacked from bottom (worst) to top (best)
SEM_BIN_ORDER       = ["Low / no semantic relevance", "Partial overlap", "Strong conceptual similarity", "Near identical"]
SEM_BIN_COLORS_HIST = ["#003F8A", "#6BB8FF", "#FF6B6B", "#8B0000"]


LLM_JUDGE_SCORE_ORDER = [1, 2, 3, 4]
LLM_JUDGE_COLORS_HIST = ["#ffffcc", "#d9f0a3", "#78c679", "#1a9641"]


def _plot_score_histogram(ax, csv_path: str, title: str = ""):
    """
    Draws two stacked bars side by side per benchmark set onto ax:
      left bar  — semantic similarity bins  (bottom = no relevance, top = near identical)
      right bar — LLM judge scores          (bottom = score 1, top = highest score)
    """
    df = pd.read_csv(csv_path).dropna(subset=["semantic_similarity", "llm_judge_score"])
    df["sem_bin"]   = df["semantic_similarity"].apply(bin_semantic)
    df["llm_score"] = df["llm_judge_score"].astype(int)

    bar_w = 0.3
    gap   = 0.04
    x     = np.arange(len(BENCHMARK_SETS_WITHIN))

    # Left stacked bar per benchmark set: semantic similarity bins
    bottoms = np.zeros(len(BENCHMARK_SETS_WITHIN))
    for label, color in zip(SEM_BIN_ORDER, SEM_BIN_COLORS_HIST):
        heights = [
            df[df["prediction_type"] == bset]["sem_bin"].eq(label).sum()
            / max(df[df["prediction_type"] == bset]["sem_bin"].count(), 1)
            for bset in BENCHMARK_SETS_WITHIN
        ]
        ax.bar(x - bar_w / 2 - gap / 2, heights, bar_w, bottom=bottoms, color=color)
        bottoms += np.array(heights)

    # Right stacked bar per benchmark set: LLM judge scores
    bottoms = np.zeros(len(BENCHMARK_SETS_WITHIN))
    for score, color in zip(LLM_JUDGE_SCORE_ORDER, LLM_JUDGE_COLORS_HIST):
        heights = [
            df[df["prediction_type"] == bset]["llm_score"].eq(score).sum()
            / max(df[df["prediction_type"] == bset]["llm_score"].count(), 1)
            for bset in BENCHMARK_SETS_WITHIN
        ]
        ax.bar(x + bar_w / 2 + gap / 2, heights, bar_w, bottom=bottoms, color=color)
        bottoms += np.array(heights)

    ax.set_xticks(x)
    ax.set_xticklabels([BENCHMARK_LABELS_WITHIN[b] for b in BENCHMARK_SETS_WITHIN], fontsize=9)
    ax.set_ylim(0, 1.05)
    if title:
        ax.set_title(title, fontsize=13)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def draw_score_histogram_grid(
    llm_csv_paths: dict,
    output_path: str = "score_histogram.png",
    suptitle: str = "",
):
    """
    llm_csv_paths: {llm_label: path to evaluation_results CSV}
    Draws one panel per LLM in a 2x2 grid (labeled A-D), each showing the
    semantic-similarity vs LLM-judge-score stacked bars, with a single legend
    shared across all panels.
    """
    llm_labels = list(llm_csv_paths.keys())
    panel_labels = ["A", "B", "C", "D"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9.5), sharey=True)

    for panel_label, ax, llm_label in zip(panel_labels, axes.flat, llm_labels):
        _plot_score_histogram(ax, llm_csv_paths[llm_label], title=llm_label)
        ax.text(-0.08, 1.0, panel_label, transform=ax.transAxes,
                fontsize=16, fontweight="bold", va="bottom", ha="left")

    for ax in axes[:, 0]:
        ax.set_ylabel("Proportion", fontsize=12)
    for ax in axes[-1, :]:
        ax.set_xlabel("Benchmark Set", fontsize=11)

    if suptitle:
        fig.suptitle(suptitle, fontsize=14, y=1.02)

    # Two-row legend: row 1 = semantic similarity, row 2 = LLM judge scores
    sem_title = Patch(facecolor="none", edgecolor="none", label="Semantic Similarity:")
    sem_handles = [Patch(facecolor=c, label=l) for l, c in zip(SEM_BIN_ORDER, SEM_BIN_COLORS_HIST)]
    sem_handles.reverse()  # Reverse order for legend to match stacked bar order
    llm_title = Patch(facecolor="none", edgecolor="none", label="LLM Judge Score:")
    llm_handles = [Patch(facecolor=c, label=f"Score {s}") for s, c in zip(LLM_JUDGE_SCORE_ORDER, LLM_JUDGE_COLORS_HIST)]
    llm_handles.reverse()  # Reverse order for legend to match stacked bar order

    leg1 = fig.legend(handles=[sem_title] + sem_handles, frameon=False, fontsize=10,
                       loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=len(sem_handles) + 1)
    fig.add_artist(leg1)
    fig.legend(handles=[llm_title] + llm_handles, frameon=False, fontsize=10,
               loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=len(llm_handles) + 1)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
for task, suffix in [("Process Names", "processNames"), ("Analytic Narratives", "analyticNarratives")]:
    llm_csv_paths = {llm: f"{base_path}/evaluation_results_{suffix}.csv" for llm, base_path in LLM_BASE_PATHS.items()}
    draw_score_histogram_grid(
        llm_csv_paths,
        output_path=f"figures/score_histogram_{suffix}.png",
        suptitle=task,
    )

# Heatmap: Rank-Based Correlation Between Semantic Similarity and LLM Judge Score

In [ ]:
from scipy.stats import spearmanr
import matplotlib.colors as mcolors

HEATMAP_LLM_ORDER = ["GPT-5.4 mini", "GPT-OSS", "Gemma4", "Mixtral"]
HEATMAP_BENCHMARK_SETS = ["full_set", "reduced_set", "noise_20", "noise_40", "noise_60", "noise_80"]
HEATMAP_BENCHMARK_LABELS = [
    "Full Set",
    "Pharmacologically\nActionable Set",
    "Noise 20%",
    "Noise 40%",
    "Noise 60%",
    "Noise 80%",
]

HEATMAP_CSV_PATHS = {
    "processNames": {
        "GPT-5.4 mini": "Outputs/azure-gpt-5.4-mini/evaluation_results_processNames.csv",
        "GPT-OSS":       "Outputs/gpt-oss:20b/evaluation_results_processNames.csv",
        "Gemma4":        "Outputs/gemma4:26b/evaluation_results_processNames.csv",
        "Mixtral":       "Outputs/mixtral-8x22b/evaluation_results_processNames.csv",
    },
    "analyticNarratives": {
        "GPT-5.4 mini": "Outputs/azure-gpt-5.4-mini/evaluation_results_analyticNarratives.csv",
        "GPT-OSS":       "Outputs/gpt-oss:20b/evaluation_results_analyticNarratives.csv",
        "Gemma4":        "Outputs/gemma4:26b/evaluation_results_analyticNarratives.csv",
        "Mixtral":       "Outputs/mixtral-8x22b/evaluation_results_analyticNarratives.csv",
    },
}

# Ordinal encoding for semantic similarity bins (matches LLM judge score scale 1–4)
SEM_BIN_CODES = {
    "Low / no semantic relevance":    1,
    "Partial overlap":                2,
    "Strong conceptual similarity":   3,
    "Near identical":                 4,
}

RHO_BOUNDARIES = np.arange(0, 1.1, 0.1)


def _sig_stars(pval: float) -> str:
    if pval < 0.001:
        return "***"
    elif pval < 0.01:
        return "**"
    elif pval < 0.05:
        return "*"
    return "ns"

def holm_adjust(p_values):
    """
    Direct Holm step-down adjustment that preserves input order.
    """
    p_values = np.asarray(p_values)
    order = np.argsort(p_values)
    adjusted_sorted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    adjusted = np.empty(len(p_values), dtype=float)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)

    return adjusted


def _compute_spearman_matrix(csv_paths: dict, llm_order: list, benchmark_sets: list):
    """
    Return rho and p-value matrices of shape (n_llms, n_benchmarks).
    Semantic similarity is binned into ordinal codes (1–4) before correlation.
    """
    n_llm = len(llm_order)
    n_bset = len(benchmark_sets)
    rho_mat = np.full((n_llm, n_bset), np.nan)
    pval_mat = np.ones((n_llm, n_bset))

    for i, llm in enumerate(llm_order):
        df = pd.read_csv(csv_paths[llm]).dropna(subset=["semantic_similarity", "llm_judge_score"])
        df["sem_bin_code"] = df["semantic_similarity"].apply(bin_semantic).map(SEM_BIN_CODES)
        for j, bset in enumerate(benchmark_sets):
            subset = df[df["prediction_type"] == bset].dropna(subset=["sem_bin_code"])
            if len(subset) >= 5:
                rho, pval = spearmanr(subset["sem_bin_code"], subset["llm_judge_score"])
                rho_mat[i, j] = rho
                pval_mat[i, j] = pval

    pval_mat = holm_adjust(pval_mat.flatten()).reshape(pval_mat.shape)

    return rho_mat, pval_mat


def draw_correlation_heatmap(
    csv_paths: dict,
    llm_order: list,
    benchmark_sets: list,
    benchmark_labels: list,
    title: str = "",
    output_path: str = "figures/correlation_heatmap.png",
):
    """
    Draw a single Spearman ρ heatmap for one task type.
    Rows = LLMs, columns = benchmark sets.
    Cell color encodes ρ in 0.1-increment bins over [0, 1] using a warm-to-cold
    colormap (red = low correlation, blue = high correlation).
    Annotation shows ρ value and significance stars.
    """
    rho_mat, pval_mat = _compute_spearman_matrix(csv_paths, llm_order, benchmark_sets)

    cmap = plt.cm.RdYlBu_r  # warm (red) at low ρ → cold (blue) at high ρ
    norm = mcolors.BoundaryNorm(boundaries=RHO_BOUNDARIES, ncolors=cmap.N)

    fig, ax = plt.subplots(figsize=(8, 4))

    im = ax.imshow(rho_mat, aspect="auto", cmap=cmap, norm=norm)

    ax.set_xticks(range(len(benchmark_sets)))
    
    ax.set_xticklabels(benchmark_labels, fontsize=10)
    ax.set_yticks(range(len(llm_order)))
    ax.set_yticklabels(llm_order, fontsize=11)
    ax.set_xlabel("Benchmark Set", fontsize=11)
    ax.set_ylabel("LLM", fontsize=11)
    if title:
        ax.set_title(title, fontsize=13, pad=10)

    for i in range(len(llm_order)):
        for j in range(len(benchmark_sets)):
            rho = rho_mat[i, j]
            pval = pval_mat[i, j]
            if np.isnan(rho):
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=9, color="gray")
                continue
            stars = _sig_stars(pval)
            text_color = "white" if rho < 0.2 or rho > 0.75 else "black"
            ax.text(
                j, i,
                f"ρ={rho:.2f}\n{stars}",
                ha="center", va="center",
                fontsize=9, color=text_color,
                fontweight="bold" if stars != "ns" else "normal",
            )

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=RHO_BOUNDARIES)
    cbar.set_label("Spearman ρ", fontsize=10)
    cbar.ax.set_yticklabels([f"{v:.1f}" for v in RHO_BOUNDARIES], fontsize=8)

    sig_note = "Significance (adjusted): * p<0.05  ** p<0.01  *** p<0.001  ns = not significant"
    fig.text(0.5, -0.1, sig_note, ha="center", fontsize=9, color="dimgray", style="italic")

    plt.tight_layout()
    plt.xticks(rotation=15)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


for task, task_title in [("processNames", "Process Names"), ("analyticNarratives", "Analytic Narratives")]:
    draw_correlation_heatmap(
        csv_paths=HEATMAP_CSV_PATHS[task],
        llm_order=HEATMAP_LLM_ORDER,
        benchmark_sets=HEATMAP_BENCHMARK_SETS,
        benchmark_labels=HEATMAP_BENCHMARK_LABELS,
        title=f"Binned Semantic Similarity vs LLM Judge Score\n{task_title}",
        output_path=f"figures/correlation_heatmap_{task}.png",
    )

# Evaluation Metric vs Incompleteness and Gene Set Size

In [ ]:
# miss_bin (quartile of missing_pct: the fraction of druggable genes withheld to
# build reduced_set) and druggable_genes (count of genes in the reduced/actionable
# gene set) are both read from the same source metadata file, and are used to bin
# "reduced_set" predictions by two orthogonal properties: how incomplete the input
# gene list is, and how large it is.
BINNED_METADATA_CSV = "Datasets/AlzKB/sampled_noise.csv"
MISS_BIN_ORDER = ["1-20%", "20-40%", "40-60%", "60-80%"]
MISS_BIN_LABEL = ["[1%, 20%)", "[20%, 40%)", "[40%, 60%)", "[60%, 80%]"]
DRUG_BIN_EDGES = [0, 10, 20, 40]
DRUG_BIN_ORDER = ["5-10", "10-20", "20-36"]
DRUG_BIN_LABEL = ["[5, 10]", "(10, 20]", "(20, 36]"]
LLM_COLORS = {
    "GPT-5.4 mini": "#648FFF",
    "GPT-OSS":       "#785EF0",
    "Gemma4":        "#DC267F",
    "Mixtral":       "#FE6100",
}
BINNED_SUB_SPACING = 0.05

def _load_binned_metadata(bin_col: str) -> pd.DataFrame:
    """
    Returns a [pathway_id, bin_col] DataFrame. miss_bin is read directly from the
    metadata CSV; drug_bin is derived from the druggable_genes count.
    """
    metadata = pd.read_csv(BINNED_METADATA_CSV)
    if bin_col == "drug_bin":
        metadata[bin_col] = pd.cut(metadata["druggable_genes"], bins=DRUG_BIN_EDGES, labels=DRUG_BIN_ORDER)
    else:
        metadata[bin_col] = metadata[bin_col].str.strip("'")
    return metadata[[bin_col]].reset_index().rename(columns={"index": "pathway_id"})

def load_binned_data(llm_csv_paths: dict, y_col: str, bin_col: str, benchmark_set: str = "reduced_set") -> dict:
    """
    llm_csv_paths: {display_label: path_to_evaluation_results_processNames.csv}
    bin_col: "miss_bin" or "drug_bin"
    Returns: {display_label: DataFrame with columns [bin_col, y_col]}
    """
    metadata = _load_binned_metadata(bin_col)
    llm_data = {}
    for label, csv_path in llm_csv_paths.items():
        df = pd.read_csv(csv_path)
        df = df[df["prediction_type"] == benchmark_set].dropna(subset=[y_col])
        df = df.merge(metadata, on="pathway_id")
        llm_data[label] = df[[bin_col, y_col]]
    return llm_data

def _plot_binned(ax, llm_data: dict, y_col: str, bin_col: str, bin_order: list, x_label: str):
    """Draw mean +/- SEM per bin, one line per LLM, onto ax."""
    llm_labels = list(llm_data.keys())
    n_llms     = len(llm_labels)
    offsets    = np.linspace(-(n_llms - 1) / 2, (n_llms - 1) / 2, n_llms) * BINNED_SUB_SPACING
    x          = np.arange(len(bin_order))

    for j, llm in enumerate(llm_labels):
        color = LLM_COLORS[llm]
        stats_by_bin = llm_data[llm].groupby(bin_col)[y_col].agg(["mean", "sem"]).reindex(bin_order)
        ax.errorbar(
            x + offsets[j], stats_by_bin["mean"], yerr=stats_by_bin["sem"],
            color=color, marker="o", ms=6, lw=2, capsize=3, label=llm,
        )

    ax.set_xticks(x)
    ax.set_xlabel(x_label, fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def draw_two_panel_binned(
    panels: list,
    y_col: str,
    y_label: str,
    output_path: str,
    orientation: str = "horizontal",
):
    """
    panels: list of two dicts, each with keys
        panel_label, llm_data, bin_col, bin_order, bin_label, x_label
    Draws panel A and B either side by side ("horizontal") or stacked
    ("vertical"), each overlaying one mean +/- SEM line per LLM, with a single
    combined legend below.
    """
    if orientation == "horizontal":
        fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    else:
        fig, axes = plt.subplots(2, 1, figsize=(7, 9))

    for ax, panel in zip(axes, panels):
        _plot_binned(ax, panel["llm_data"], y_col, panel["bin_col"], panel["bin_order"], panel["x_label"])
        ax.set_xticklabels(panel["bin_label"], fontsize=11)
        ax.set_ylabel(y_label, fontsize=12)
        ax.text(-0.08, 1.03, panel["panel_label"], transform=ax.transAxes,
                fontsize=16, fontweight="bold", va="bottom", ha="left")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, fontsize=10,
               loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=len(labels))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


## Semantic Similarity

In [ ]:
processname_csv_paths = {llm: f"{base}/evaluation_results_processNames.csv" for llm, base in LLM_BASE_PATHS.items()}

sem_data_miss = load_binned_data(processname_csv_paths, "semantic_similarity", bin_col="miss_bin")
sem_data_drug = load_binned_data(processname_csv_paths, "semantic_similarity", bin_col="drug_bin")

draw_two_panel_binned(
    panels=[
        {"panel_label": "A", "llm_data": sem_data_miss, "bin_col": "miss_bin",
         "bin_order": MISS_BIN_ORDER, "bin_label": MISS_BIN_LABEL, "x_label": "Incompleteness"},
        {"panel_label": "B", "llm_data": sem_data_drug, "bin_col": "drug_bin",
         "bin_order": DRUG_BIN_ORDER, "bin_label": DRUG_BIN_LABEL, "x_label": "Druggable Genes"},
    ],
    y_col="semantic_similarity",
    y_label="Semantic Similarity",
    orientation="horizontal",
    output_path="figures/incompleteness_size_semantic_similarity_processNames.png",
)


In [ ]:
analyticnarrative_csv_paths = {llm: f"{base}/evaluation_results_analyticNarratives.csv" for llm, base in LLM_BASE_PATHS.items()}

sem_data_miss = load_binned_data(analyticnarrative_csv_paths, "semantic_similarity", bin_col="miss_bin")
sem_data_drug = load_binned_data(analyticnarrative_csv_paths, "semantic_similarity", bin_col="drug_bin")

draw_two_panel_binned(
    panels=[
        {"panel_label": "A", "llm_data": sem_data_miss, "bin_col": "miss_bin",
         "bin_order": MISS_BIN_ORDER, "bin_label": MISS_BIN_LABEL, "x_label": "Incompleteness"},
        {"panel_label": "B", "llm_data": sem_data_drug, "bin_col": "drug_bin",
         "bin_order": DRUG_BIN_ORDER, "bin_label": DRUG_BIN_LABEL, "x_label": "Druggable Genes"},
    ],
    y_col="semantic_similarity",
    y_label="Semantic Similarity",
    orientation="horizontal",
    output_path="figures/incompleteness_size_semantic_similarity_analyticNarrative.png",
)


## LLM Judge Score

In [ ]:
def _plot_score_bars(ax, df: pd.DataFrame, bin_col: str, bin_order: list, bin_label: list, x_label: str):
    """Draw one stacked bar of judge score proportions per bin onto ax."""
    x     = np.arange(len(bin_order))
    bar_w = 0.6
    scores = df["llm_judge_score"].astype(int)

    bottoms = np.zeros(len(bin_order))
    for score, color in zip(LLM_JUDGE_SCORE_ORDER, LLM_JUDGE_COLORS_HIST):
        heights = [
            scores[df[bin_col] == b].eq(score).sum() / max((df[bin_col] == b).sum(), 1)
            for b in bin_order
        ]
        ax.bar(x, heights, bar_w, bottom=bottoms, color=color)
        bottoms += np.array(heights)

    ax.set_xticks(x)
    ax.set_xticklabels(bin_label, fontsize=11)
    ax.set_xlabel(x_label, fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def draw_llm_score_bar_grid(llm_data_miss: dict, llm_data_drug: dict, output_path: str):
    """
    llm_data_miss / llm_data_drug: {llm_label: DataFrame[bin_col, llm_judge_score]}
    from load_binned_data. Draws a 2x4 grid: one column per LLM, top row shows
    Incompleteness bins and bottom row shows Gene Set Size (Druggable Genes)
    bins, each as stacked LLM judge score proportions, with a single legend
    shared across all panels.
    """
    llm_labels = list(llm_data_miss.keys())
    fig, axes = plt.subplots(2, len(llm_labels), figsize=(4.1 * len(llm_labels), 7), sharey=True)

    for col, llm in enumerate(llm_labels):
        _plot_score_bars(axes[0, col], llm_data_miss[llm], "miss_bin", MISS_BIN_ORDER, MISS_BIN_LABEL, "Incompleteness")
        _plot_score_bars(axes[1, col], llm_data_drug[llm], "drug_bin", DRUG_BIN_ORDER, DRUG_BIN_LABEL, "Druggable Genes")
        axes[0, col].set_title(llm, fontsize=13)

    axes[0, 0].set_ylabel("Proportion", fontsize=11)
    axes[1, 0].set_ylabel("Proportion", fontsize=11)

    legend_handles = [Patch(facecolor=c, label=f"Score {s}") for s, c in zip(LLM_JUDGE_SCORE_ORDER, LLM_JUDGE_COLORS_HIST)]
    legend_handles.reverse()
    fig.legend(handles=legend_handles, frameon=False, fontsize=10,
               loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=len(legend_handles))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
miss_data = load_binned_data(processname_csv_paths, "llm_judge_score", bin_col="miss_bin")
drug_data = load_binned_data(processname_csv_paths, "llm_judge_score", bin_col="drug_bin")

draw_llm_score_bar_grid(
    llm_data_miss=miss_data,
    llm_data_drug=drug_data,
    output_path="figures/incompleteness_size_judge_score_processNames.png",
)


In [ ]:
miss_data = load_binned_data(analyticnarrative_csv_paths, "llm_judge_score", bin_col="miss_bin")
drug_data = load_binned_data(analyticnarrative_csv_paths, "llm_judge_score", bin_col="drug_bin")

draw_llm_score_bar_grid(
    llm_data_miss=miss_data,
    llm_data_drug=drug_data,
    output_path="figures/incompleteness_size_judge_score_analyticNarrative.png",
)
